# Kerala helpdesk: canonical services + held-out eval with Gemini (tasks B and A)

1. Groups the ~1,250 charter rows in `data/docs.jsonl` into services (no LLM).
2. Asks Gemini, in batches, for each group: is it a service, a clean Malayalam title, synonyms, facts common to the
   rows, each panchayat's fee and time, and 8 citizen questions. Every number it writes is checked against the rows.
3. Writes `data/canon.jsonl` (one passage per service, 4 questions indexed) and `eval/synth_questions.jsonl`
   (the other 4, never indexed, plus Tier-1 and out-of-scope questions).
4. Runs `eval/run_synth.py`: baseline (today's rows) vs canonical services, on held-out questions.

**Setup**: Add-ons → Secrets → add `GEMINI_API_KEY` and attach it to this notebook. Settings → Internet on. CPU is enough.
Run with **Save Version → Save & Run All**. The run stops itself after `BUDGET_MIN` minutes (default 120) and still writes outputs. If the daily quota runs out, the finished groups are kept in
`ingest/canon_cache.jsonl`: add this notebook's output as Input next day and run again; it continues.

Only public charter text is sent. Free-tier Gemini prompts may be used by Google to improve its products:
never send user logs or audio this way.

In [ ]:
# ---- CONFIG ----
REPO_URL  = "https://github.com/cksajil/kerala-voice-helpdesk"
BRANCH    = ""                          # empty = default branch
MODEL     = "gemini-flash-lite-latest"  # see the model list printed below; gemini-flash-latest is better but has a lower daily quota
BATCH     = 8                           # groups per request
MAX_REQ   = 400                         # stop after this many requests in one run
PAUSE_S   = 4.0                         # seconds between requests (free tier is rate limited per minute)
BUDGET_MIN = 120                        # stop starting new requests after this many minutes (outputs are still written)
TIMEOUT_S = 180                         # abandon one request after this many seconds
DRY_RUN   = False                       # True: only print the number of groups and requests

In [ ]:
import os, subprocess, sys, glob, shutil, zipfile, pathlib, json
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", *(["-b", BRANCH] if BRANCH else []), REPO_URL, "repo"], check=True)
os.chdir("/kaggle/working/repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "rank_bm25", "langgraph", "feedparser"], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
print(sum(1 for _ in open("data/docs.jsonl")), "passages in data/docs.jsonl")

In [ ]:
# Which models does this key reach? (names change; pick MODEL from here)
from google import genai
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
names = sorted(m.name.split("/")[-1] for m in client.models.list())
print([n for n in names if "flash" in n or "gemma" in n])

In [ ]:
# Resume: bring back the cache from a previous version of this notebook (added as Input).
for c in glob.glob("/kaggle/input/**/canon_cache.jsonl", recursive=True):
    shutil.copy(c, "ingest/canon_cache.jsonl"); print("restored", c, sum(1 for _ in open(c)), "answers")

In [ ]:
args = ["--model", MODEL, "--batch", str(BATCH), "--max-requests", str(MAX_REQ), "--pause", str(PAUSE_S),
        "--budget-min", str(BUDGET_MIN), "--timeout", str(TIMEOUT_S)]
!python -u -W ignore -m ingest.canon {" ".join(args)} {"--dry-run" if DRY_RUN else ""}


In [ ]:
# Look at a few canonical services before trusting them.
canon = [json.loads(l) for l in open("data/canon.jsonl", encoding="utf-8")]
for d in canon[:8]:
    print(d["service"], "|", d["title"], "|", len(d["members"]), "rows,", len(d["lsgis"]), "panchayats")
    print("   ", d["text"][:220]); print("    Q:", d["questions"][:3]); print()

In [ ]:
# Held-out eval: today's rows vs canonical services (test split; --split all for everything)
!python -W ignore -m eval.run_synth --split test --show 10

In [ ]:
out = pathlib.Path("/kaggle/working/canon_out"); shutil.rmtree(out, ignore_errors=True)
for k in ["data/canon.jsonl", "eval/synth_questions.jsonl", "eval/synth_report.json", "ingest/canon_cache.jsonl"]:
    if os.path.exists(k):
        (out / k).parent.mkdir(parents=True, exist_ok=True); shutil.copy(k, out / k)
shutil.make_archive("/kaggle/working/canon_out", "zip", out)
print("download canon_out.zip from Output, unzip in the repo root")